# 判断準備用の7表を確認する

これは比較用派生データです。元の13表・30分市場表・元頻度需要・固定案Aはそのままです。
主分析の頻度・最終headroom・法人帰属・供給者分類は確定していません。

`.venv`を選んで「すべて実行」すると、各表の行数・全列名・主要列の実データが表示されます。
末尾で表・日付・地域を指定し、全列のページ閲覧ができます。原ファイルへの書込みや補完は行いません。

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import HTML, display

# worktreeからも、実データのある共有repoを探す。
relative_data = Path("data/processed/comparison/20261016_decision_preparation_v2")
repo = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "AGENTS.md").exists() and (p / relative_data).is_dir()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError(
        "共有repoのdata/が見つかりません。このrepo内から実行してください。"
        "GitHubからのcloneだけでは実データは含まれません。"
    )
data_dir = repo / relative_data
print(f"読取先: {data_dir}")

In [ ]:
TABLES = {
    "hourly": {
        "title": "1時間価格・需要比較",
        "filename": "market_hourly_price_demand_FY2025.parquet",
        "columns": [
            "area",
            "delivery_date",
            "hour_of_delivery",
            "halfhour_1__area_price",
            "halfhour_2__area_price",
            "area_price__time_mean",
            "area_price__minimum",
            "area_price__maximum",
            "area_price__second_minus_first",
            "demand_source__demand_hourly_csv_mwh",
        ],
    },
    "system": {
        "title": "系統空容量の方向別比較",
        "filename": "system_capacity_dayahead2_planned_direction_comparison_FY2025.parquet",
        "columns": [
            "delivery_date",
            "period",
            "link_name_raw",
            "direction_raw",
            "available_capacity_mw",
            "available_capacity_direction_display_mw",
            "sending_area",
            "receiving_area",
            "receiving_minus_sending_price",
        ],
    },
    "nextday2": {
        "title": "予備率連系線・翌々日比較",
        "filename": "reserve_interconnector_nextday2_direction_comparison_FY2025.parquet",
        "columns": [
            "delivery_date",
            "period",
            "link_name_raw",
            "forward_available_capacity_mw",
            "forward_available_capacity_comparison_mw",
            "forward__receiving_minus_sending_price",
            "forward__price_comparison_status",
        ],
    },
    "latest": {
        "title": "予備率連系線・翌日当日比較",
        "filename": "reserve_interconnector_latest_direction_comparison_FY2025.parquet",
        "columns": [
            "delivery_date",
            "period",
            "link_name_raw",
            "forward_available_capacity_mw",
            "forward_available_capacity_comparison_mw",
            "forward__receiving_minus_sending_price",
            "forward__price_comparison_status",
        ],
    },
    "units": {
        "title": "全491unitの台帳事業者参照",
        "filename": "observed_unit_business_evidence_FY2025.parquet",
        "columns": [
            "area",
            "plant_name",
            "unit_name",
            "generation_sum_kwh",
            "hjks_business_name_unique_reference",
            "hjks_business_name_reference_status",
            "hjks_business_name_candidates_json",
            "fy2025_asset_business_relationship_verified",
        ],
    },
    "candidates": {
        "title": "HJKS候補行と原セル",
        "filename": "hjks_business_candidate_rows_FY2025.parquet",
        "columns": [
            "area",
            "asset_key_candidate",
            "hjks_snapshot_row_id",
            "business_name",
            "equipment_reference_status",
            "candidate_equipment_route",
            "observed_generation_kwh_candidate_range",
        ],
    },
    "names": {
        "title": "制度上の法人との名称比較",
        "filename": "business_regulatory_name_comparison_FY2025.parquet",
        "columns": [
            "hjks_business_name_raw",
            "regulatory_entity",
            "regulatory_market_area",
            "name_comparison",
            "asset_attribution_established",
            "supplier_classification_assigned",
        ],
    },
}


def main_columns(key):
    return TABLES[key]["columns"]

In [ ]:
def read_page(
    key,
    *,
    area=None,
    delivery_date=None,
    period=None,
    hour=None,
    link=None,
    offset=0,
    limit=20,
    columns=None,
):
    if offset < 0 or limit < 1:
        raise ValueError("開始行は0以上、表示件数は1以上にしてください。")
    dataset = ds.dataset(data_dir / TABLES[key]["filename"], format="parquet")
    selected_columns = main_columns(key) if columns == "main" else columns
    expression = None
    for field, value in [
        ("area", area),
        ("delivery_date", delivery_date),
        ("period", period),
        ("link_name_raw", link),
        ("hour_of_delivery", hour),
    ]:
        if value is None or value == "":
            continue
        if field not in dataset.schema.names:
            raise ValueError(f"この表に{field}はありません。該当フィルターを空にしてください。")
        if field == "delivery_date" and isinstance(value, str):
            value = date.fromisoformat(value)
        condition = ds.field(field) == pa.scalar(value, type=dataset.schema.field(field).type)
        expression = condition if expression is None else expression & condition
    matched = dataset.count_rows(filter=expression)
    chunks = []
    skipped = 0
    remaining = limit
    for batch in dataset.to_batches(columns=selected_columns, filter=expression, batch_size=2048):
        if skipped + batch.num_rows <= offset:
            skipped += batch.num_rows
            continue
        begin = max(0, offset - skipped)
        part = batch.slice(begin, min(remaining, batch.num_rows - begin))
        chunks.append(part)
        remaining -= part.num_rows
        skipped += batch.num_rows
        if remaining == 0:
            break
    schema = (
        dataset.schema
        if selected_columns is None
        else pa.schema([dataset.schema.field(c) for c in selected_columns])
    )
    frame = pa.Table.from_batches(chunks, schema=schema).to_pandas()
    frame.attrs["matched_rows"] = matched
    frame.attrs["offset"] = offset
    return frame


def show_frame(frame):
    print(
        f"条件に合う全{frame.attrs['matched_rows']:,}行のうち、"
        f"{frame.attrs['offset']:,}行目から{len(frame):,}行を表示 / {len(frame.columns)}列"
    )
    display(
        HTML(
            '<div style="overflow:auto;max-height:480px">'
            + frame.to_html(index=False, max_rows=None, max_cols=None)
            + "</div>"
        )
    )


def show_table(key):
    info = TABLES[key]
    stored = pq.ParquetFile(data_dir / info["filename"])
    print(f"{info['filename']}\n{stored.metadata.num_rows:,}行・{len(stored.schema_arrow)}列")
    fields = pd.DataFrame([{"列名": f.name, "型": str(f.type)} for f in stored.schema_arrow])
    display(
        HTML(
            "<details><summary>全列名と型を開く</summary>"
            '<div style="overflow:auto;max-height:360px">'
            + fields.to_html(index=False, escape=True)
            + "</div></details>"
        )
    )
    frame = read_page(key, limit=10, columns="main")
    show_frame(frame)
    return frame

In [ ]:
inventory = []
for key, info in TABLES.items():
    stored = pq.ParquetFile(data_dir / info["filename"])
    inventory.append(
        {
            "選択ID": key,
            "表": info["title"],
            "行数": stored.metadata.num_rows,
            "列数": len(stored.schema_arrow),
        }
    )
display(pd.DataFrame(inventory))
previews = {}

## 1時間価格・需要比較

前半・後半から等時間算術平均等を追加。需要は1時間の原値で、30分へ補間していません。

In [ ]:
previews["hourly"] = show_table("hourly")

## 系統空容量の方向別比較

系統CSVの空容量だけ順×1・逆×−1の表示値を別列に追加。原符号も保持。単一地域間の価格差は記述用です。

In [ ]:
previews["system"] = show_table("system")

## 予備率連系線・翌々日比較

別欄の公表数量は無変換。予想潮流を実潮流としません。多地域フェンスや方向不明の価格差はnull。

In [ ]:
previews["nextday2"] = show_table("nextday2")

## 予備率連系線・翌日当日比較

計画断面・出典・更新時点は元列に残っています。地域輸入能力へ合算しません。

In [ ]:
previews["latest"] = show_table("latest")

## 全491unitの台帳事業者参照

現在の参照snapshot「発電事業者」欄を整理。保留・候補のみも保持。発電量は観測unitの量で、法人帰属量ではありません。

In [ ]:
previews["units"] = show_table("units")

## HJKS候補行と原セル

605候補行。発電実績ではありません。同じunitが複数行に関係するため候補発電量を単純加算しません。

In [ ]:
previews["candidates"] = show_table("candidates")

## 制度上の法人との名称比較

名称の一致・字形差・不一致のみ。グループ関係による割当、資産帰属やdominant/fringe判定はしていません。

In [ ]:
previews["names"] = show_table("names")

## 自由に閲覧

「全列」で元の価格列・需要列・出典やQA列も見られます。新しい表を選ぶと絞り込みを空にします。
1時間表にはperiodでなくhour_of_deliveryがあります。unit表にperiod、法人名比較表にareaはありません。
使えない条件を指定した場合は理由を表示します。表示の抜粋は元ファイルの削除・研究サンプルの除外ではありません。

In [ ]:
# ipywidgetsが使えない環境でも、次のコードセルで全表・全列を閲覧できます。
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("対話欄が使えないため、次のセルの選択ID・条件を編集して閲覧してください。")

if widgets is not None:
    table_choice = widgets.Dropdown(
        options=[(info["title"], key) for key, info in TABLES.items()],
        description="表",
        layout=widgets.Layout(width="650px"),
    )
    area_input = widgets.Text(description="地域", placeholder="空欄＝全地域。例：東京")
    date_input = widgets.Text(description="受渡日", placeholder="YYYY-MM-DD または空欄")
    period_input = widgets.Text(description="period", placeholder="1〜48 または空欄")
    hour_input = widgets.Text(description="hour", placeholder="0〜23 または空欄")
    link_input = widgets.Text(description="連系線", placeholder="元の名称と完全一致 または空欄")
    view_choice = widgets.ToggleButtons(options=[("全列", "all"), ("主要列", "main")])
    row_offset = widgets.IntText(value=0, description="開始行")
    page_size = widgets.BoundedIntText(value=20, min=1, max=500, description="表示件数")
    show_button = widgets.Button(description="表示", button_style="primary")
    previous_button = widgets.Button(description="前のページ")
    next_button = widgets.Button(description="次のページ")
    viewer_output = widgets.Output()

    def draw_page(_=None):
        global selected_page
        with viewer_output:
            viewer_output.clear_output(wait=True)
            try:
                selected_page = read_page(
                    table_choice.value,
                    area=area_input.value.strip() or None,
                    delivery_date=date_input.value.strip() or None,
                    period=int(period_input.value) if period_input.value.strip() else None,
                    hour=int(hour_input.value) if hour_input.value.strip() else None,
                    link=link_input.value.strip() or None,
                    offset=row_offset.value,
                    limit=page_size.value,
                    columns=None if view_choice.value == "all" else "main",
                )
                show_frame(selected_page)
            except (ValueError, KeyError, pa.ArrowException) as error:
                selected_page = None
                print(f"表示条件を確認してください: {error}")

    def change_page(step):
        row_offset.value = max(0, row_offset.value + step * page_size.value)
        draw_page()

    def reset_filters(_):
        area_input.value = ""
        date_input.value = ""
        period_input.value = ""
        link_input.value = ""
        hour_input.value = ""
        row_offset.value = 0

    table_choice.observe(reset_filters, names="value")
    show_button.on_click(draw_page)
    previous_button.on_click(lambda _: change_page(-1))
    next_button.on_click(lambda _: change_page(1))
    display(
        widgets.VBox(
            [
                table_choice,
                widgets.HBox([area_input, date_input]),
                widgets.HBox([period_input, hour_input]),
                link_input,
                view_choice,
                widgets.HBox([row_offset, page_size]),
                widgets.HBox([show_button, previous_button, next_button]),
                viewer_output,
            ]
        )
    )
    draw_page()

対話欄が使えない場合は、次のセルの選択IDや条件を変えて実行してください。
`columns=None`は全列です。1時間表はhour（0〜23）、30分表はperiod（1〜48）で絞れます。
このノートブックは計算や再取得をせず、保存済みデータを読むだけです。

In [ ]:
manual_page = read_page(
    "hourly", area="東京", delivery_date="2026-03-31", offset=0, limit=24, columns="main"
)
show_frame(manual_page)

全行をデータビューアーで確認する場合は、必要な1表だけを読みます。

```python
full_table = pq.read_table(data_dir / TABLES["hourly"]["filename"]).to_pandas()
# 例：表示用にhour_of_deliveryで絞る。原ファイルは変更しません。
display(full_table[full_table.hour_of_delivery.eq(23)])
```

[短い判断資料](../reports/11_decision_preparation/README.md) / [定義と実行仕様](../docs/decision_preparation.md)
